# Track 5: Group Relative Policy Optimization (GRPO) for Mathematical Reasoning

This notebook demonstrates fine-tuning **Qwen2.5-3B-Instruct** on mathematical reasoning problems from the `openai/gsm8k` dataset using **Group Relative Policy Optimization (GRPO)**. We configure reinforcement learning rewards based on format correctness and solution accuracy, and leverage **vLLM** inside `GRPOTrainer` to accelerate policy rollout generation.

## 1. Setup Environment and Imports
We load libraries and establish system prompt instructions enforcing a strict XML reasoning and answer format.

In [1]:
import os
import sys
import torch
import warnings
import re
# Patch all TRL import helper functions to return boolean instead of tuple, bypassing import bugs
import trl.import_utils as utils
for name, attr in list(vars(utils).items()):
    if name.startswith('is_') and name.endswith('_available') and callable(attr):
        def make_wrapper(func):
            return lambda *a, **k: func(*a, **k)[0] if isinstance(func(*a, **k), tuple) else func(*a, **k)
        setattr(utils, name, make_wrapper(attr))
utils._vllm_available = False

from datasets import load_dataset
from transformers import AutoTokenizer
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print(f'Using device: cuda | Dtype: {compute_dtype}')

W0721 08:36:15.561000 145499 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


W0721 08:36:15.577000 145499 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Using device: cuda | Dtype: torch.bfloat16


## 2. Dataset Preparation (GSM8K)
We load the GSM8K dataset. We format each question with a system instruction that instructs the model to put its reasoning inside `<reasoning>` tags and the final numeric answer inside `<answer>` tags.

In [2]:
SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a question, and the Assistant solves it.\n"
    "The assistant first thinks about the reasoning process in the mind and then provides the user with the answer.\n"
    "The reasoning process and answer are enclosed within tags. The answer must be a single integer.\n"
    "Example:\n"
    "<reasoning>\n"
    "We know that 2 + 2 = 4.\n"
    "</reasoning>\n"
    "<answer>4</answer>"
)

def extract_hash_answer(text: str) -> str | None:
    if '####' not in text:
        return None
    return text.split('####')[1].strip()

dataset = load_dataset('openai/gsm8k', 'main', split='train')
# Select small subset of 300 prompts for fast notebook execution
dataset = dataset.shuffle(seed=42).select(range(300))

def format_gsm8k(example):
    return {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': example['question']}
        ],
        'answer': extract_hash_answer(example['answer'])
    }

gsm8k_train = dataset.map(format_gsm8k)
print('Sample Prompt Structure Preview:\n', gsm8k_train[0]['prompt'])

Sample Prompt Structure Preview:
 [{'role': 'system', 'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it.\nThe assistant first thinks about the reasoning process in the mind and then provides the user with the answer.\nThe reasoning process and answer are enclosed within tags. The answer must be a single integer.\nExample:\n<reasoning>\nWe know that 2 + 2 = 4.\n</reasoning>\n<answer>4</answer>'}, {'role': 'user', 'content': 'Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh have?'}]


## 3. Define RL Rewards
We define two reward functions:
1. **Format Reward**: Returns `1.0` if the output strictly matches the `<reasoning>...</reasoning>\n<answer>...</answer>` tags.
2. **Correctness Reward**: Returns `2.0` if the extracted answer matches the ground truth, and `0.0` otherwise.

In [3]:
def extract_last_xml_answer(text, start_tag='<answer>', end_tag='</answer>'):
    pattern = re.escape(start_tag) + r'(.*?)' + re.escape(end_tag)
    matches = re.findall(pattern, text, re.DOTALL)
    if matches:
        answer = matches[-1]
        answer = re.sub(r'[%$]', '', answer).strip()
        return answer
    return ''

def format_reward_func(completions, **kwargs):
    pattern = r'^<reasoning>[\s\S]*?<\/reasoning>\s*<answer>[\s\S]*?<\/answer>$'
    responses = [completion[0]['content'] for completion in completions]
    rewards = [1.0 if re.match(pattern, response) else 0.0 for response in responses]
    return rewards

def correctness_reward_func(completions, answer, **kwargs):
    responses = [completion[0]['content'] for completion in completions]
    extracted = [extract_last_xml_answer(response) for response in responses]
    rewards = [2.0 if ext == ans else 0.0 for ext, ans in zip(extracted, answer)]
    return rewards

## 4. Run GRPOTrainer using vLLM
We load the tokenizer and define the LoRA parameters. We use `vLLM` inside the trainer with a configured device and VRAM footprint to scale policy rollouts rapidly.

In [4]:
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

peft_config = LoraConfig(
    lora_alpha=64,
    lora_dropout=0.0,
    r=64,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

training_args = GRPOConfig(
    use_vllm=False,
    learning_rate=1e-5,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    beta=0.005,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    bf16=(compute_dtype == torch.bfloat16),
    fp16=(compute_dtype == torch.float16),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    gradient_accumulation_steps=4,
    per_device_train_batch_size=4,
    num_generations=4,
    temperature=0.5,
    max_prompt_length=256,
    max_completion_length=256,
    max_steps=50,
    logging_steps=10,
    save_steps=50,
    max_grad_norm=0.1,
    report_to='none',
    output_dir='qwen2.5-3b-grpo-output',
)

from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=compute_dtype,
    device_map='auto'
)
if not hasattr(model, 'warnings_issued'):
    model.warnings_issued = {}

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[correctness_reward_func, format_reward_func],
    args=training_args,
    train_dataset=gsm8k_train,
    peft_config=peft_config,
)

trainer.train()

merged_model = trainer.model.merge_and_unload()
tokenizer.save_pretrained('qwen2.5-3b-grpo-adapter')
merged_model.save_pretrained('qwen2.5-3b-grpo-adapter')
print('GRPO training completed and model saved!')

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


[transformers] Passing `generation_config` together with generation-related arguments=({'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Step,Training Loss
10,0.012518
20,0.014248
30,0.039114
40,-0.000279
50,0.014565


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

GRPO training completed and model saved!


## 5. Inference Verification
We query our trained model to verify that it generates reasoning and mathematical solutions structured under correct tags.

In [5]:
test_question = "Natalia sold clips to 48 of her friends in April, and then half as many in May. How many clips did Natalia sell in total?"
messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': test_question}
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt')
input_ids = inputs if isinstance(inputs, torch.Tensor) else inputs['input_ids']
input_ids = input_ids.to('cuda')

merged_model.eval()
with torch.no_grad():
    outputs = merged_model.generate(
        input_ids=input_ids,
        max_new_tokens=256,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
print('Question:', test_question)
print('\nModel response:\n', response)

Question: Natalia sold clips to 48 of her friends in April, and then half as many in May. How many clips did Natalia sell in total?

Model response:
 <reasoning>
Natalia sold 48 clips in April. In May, she sold half as many, which is 48 / 2 = 24 clips. The total number of clips sold is the sum of April's sales and May's sales, which is 48 + 24 = 72 clips.
</reasoning>
<answer>72</answer>
